In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel, SwinModel

# Initialize models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model = BertModel.from_pretrained("bert-base-uncased").to(device)

# Load the Swin model and fine-tuned weights
swin_model = SwinModel.from_pretrained("microsoft/swin-base-patch4-window7-224-in22k").to(device)
swin_model.load_state_dict(torch.load("Data for Swin/swin_model_best.pth", map_location=device))

# Define transformation layers
linear_layer = nn.Linear(768, 128).to(device)
reduce_to_768 = nn.Linear(1024, 768).to(device)  # For reducing from 1024 to 768 at the end

# Tokenizer for input words
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def generate_swin_embedding(word):
    """
    Generate a Swin embedding for a single word.
    
    Args:
    - word (str): The word to generate the embedding for.

    Returns:
    - torch.Tensor: A tensor of shape (1, 768) representing the Swin embedding for the word.
    """
    # Tokenize and get BERT embedding
    inputs = tokenizer(word, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
        last_hidden = outputs.last_hidden_state                    # (1, seq_len, 768)
        mask = inputs['attention_mask'].unsqueeze(-1)              # (1, seq_len, 1)
        word_embedding = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1)  # (1, 768)

    # Transform to match Swin requirements
    swin_input = linear_layer(word_embedding)  # Shape: (1, 128)
    swin_input = swin_input.unsqueeze(0).repeat(1, 56 * 56, 1).view(1, 56 * 56, 128)

    # Process with Swin model
    with torch.no_grad():
        swin_output = swin_model.encoder(hidden_states=swin_input, input_dimensions=(56, 56))

    # Pool and reduce dimensions
    pooled_output = swin_output[0].mean(dim=1)  # Shape: (1, 1024)
    final_swin_embedding = reduce_to_768(pooled_output)  # Shape: (1, 768)

    return final_swin_embedding

C:\Users\aadya\AppData\Local\Temp\ipykernel_8348\3970516224.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  swin_model.load_state_dict(torch.load("Data for Swin/swin_mo

In [19]:
import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi  # Ensure to install with `pip install rank-bm25`
import torch
from transformers import BertTokenizer, BertModel

# Load metadata from the CSV file
metadata = pd.read_csv('metadata.csv')

# Load the FAISS index
faiss_index = faiss.read_index('Data for Swin/swin_avg_document_embeddings.index')

# Check the columns in the metadata to confirm
print(metadata.columns)

# Combine title, snippet, and paragraphs into one searchable field (for sparse search)
# Here we assume your metadata does not have 'Title', 'Snippet', 'Paragraphs' directly,
# but includes fields like 'query', 'rank', and 'keyword' as per your previous code.
# If 'Title', 'Snippet', and 'Paragraphs' were part of `metadata`, add them back.
metadata['combined_text'] = metadata['query']  # Use query or relevant fields for combined text

# Initialize TF-IDF vectorizer and BM25 for keyword matching
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(metadata['combined_text'])
tokenized_corpus = [doc.split(" ") for doc in metadata['combined_text']]
bm25 = BM25Okapi(tokenized_corpus)

# Sparse retrieval function (TF-IDF + BM25)
def sparse_retrieval(query, top_k=5):
    query_tfidf = tfidf_vectorizer.transform([query])
    tfidf_scores = np.dot(query_tfidf, tfidf_matrix.T).toarray().flatten()
    tokenized_query = query.split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)
    combined_scores = 0.5 * tfidf_scores + 0.5 * bm25_scores
    top_indices = np.argsort(combined_scores)[::-1][:top_k]
    return top_indices

# Dense retrieval function (FAISS)
def dense_retrieval(query_embedding, top_k=5):
    query_embedding_np = query_embedding.detach().cpu().numpy().astype('float32')  # ✅ convert to NumPy
    distances, indices = faiss_index.search(query_embedding_np, top_k)
    return indices.flatten()

# Hybrid retrieval (sparse + dense)
def hybrid_retrieval(query_embedding, query, top_k=5):
    sparse_results = sparse_retrieval(query, top_k)
    dense_results = dense_retrieval(query_embedding, top_k)
    
    # Combine results with priority to dense
    combined_indices = np.concatenate([dense_results, sparse_results])
    unique_indices = np.unique(combined_indices)

    # Get the top k results prioritizing dense results first
    final_indices = list(set(dense_results) | set(unique_indices))[:top_k]

    # Retrieve metadata for the top results
    results = metadata.iloc[final_indices].to_dict(orient='records')
    return final_indices, results  # 👈 return indices too

Index(['query', 'rank', 'type', 'keyword', 'title'], dtype='object')


In [31]:
# Example usage
query = "Dimensionality Reduction"
query_embedding = generate_swin_embedding(query)
# Call retrieval
indices, results = hybrid_retrieval(query_embedding, query)

# Output results with indices
for idx, res in zip(indices, results):
    print(f"Index: {idx} -> Result: {res}")

Index: 64 -> Result: {'query': 'data science', 'rank': 8.0, 'type': 'article', 'keyword': 'data science', 'title': 'What Is Data Science?', 'combined_text': 'data science'}
Index: 37 -> Result: {'query': 'machine learning', 'rank': 9.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'What is Machine Learning? Types & Uses', 'combined_text': 'machine learning'}
Index: 38 -> Result: {'query': 'deep learning', 'rank': 1.0, 'type': 'article', 'keyword': 'deep learning', 'title': 'What is Deep Learning? - AWS', 'combined_text': 'deep learning'}
Index: 31 -> Result: {'query': 'machine learning', 'rank': 3.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'Machine Learning Tutorial', 'combined_text': 'machine learning'}
Index: 85 -> Result: {'query': 'Chapter Title', 'rank': nan, 'type': 'textbook', 'keyword': 'Chapter Keywords', 'title': 'Chapter Title', 'combined_text': 'Chapter Title'}


In [32]:
# Example usage
query = "Agentic AI"
query_embedding = generate_swin_embedding(query)
# Call retrieval
indices, results = hybrid_retrieval(query_embedding, query)

# Output results with indices
for idx, res in zip(indices, results):
    print(f"Index: {idx} -> Result: {res}")

Index: 64 -> Result: {'query': 'data science', 'rank': 8.0, 'type': 'article', 'keyword': 'data science', 'title': 'What Is Data Science?', 'combined_text': 'data science'}
Index: 37 -> Result: {'query': 'machine learning', 'rank': 9.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'What is Machine Learning? Types & Uses', 'combined_text': 'machine learning'}
Index: 38 -> Result: {'query': 'deep learning', 'rank': 1.0, 'type': 'article', 'keyword': 'deep learning', 'title': 'What is Deep Learning? - AWS', 'combined_text': 'deep learning'}
Index: 31 -> Result: {'query': 'machine learning', 'rank': 3.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'Machine Learning Tutorial', 'combined_text': 'machine learning'}
Index: 85 -> Result: {'query': 'Chapter Title', 'rank': nan, 'type': 'textbook', 'keyword': 'Chapter Keywords', 'title': 'Chapter Title', 'combined_text': 'Chapter Title'}


In [33]:
# Example usage
query = "Generative AI"
query_embedding = generate_swin_embedding(query)
# Call retrieval
indices, results = hybrid_retrieval(query_embedding, query)

# Output results with indices
for idx, res in zip(indices, results):
    print(f"Index: {idx} -> Result: {res}")

Index: 64 -> Result: {'query': 'data science', 'rank': 8.0, 'type': 'article', 'keyword': 'data science', 'title': 'What Is Data Science?', 'combined_text': 'data science'}
Index: 37 -> Result: {'query': 'machine learning', 'rank': 9.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'What is Machine Learning? Types & Uses', 'combined_text': 'machine learning'}
Index: 38 -> Result: {'query': 'deep learning', 'rank': 1.0, 'type': 'article', 'keyword': 'deep learning', 'title': 'What is Deep Learning? - AWS', 'combined_text': 'deep learning'}
Index: 31 -> Result: {'query': 'machine learning', 'rank': 3.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'Machine Learning Tutorial', 'combined_text': 'machine learning'}
Index: 85 -> Result: {'query': 'Chapter Title', 'rank': nan, 'type': 'textbook', 'keyword': 'Chapter Keywords', 'title': 'Chapter Title', 'combined_text': 'Chapter Title'}


In [34]:
# Example usage
query = "Feature Recognition"
query_embedding = generate_swin_embedding(query)
# Call retrieval
indices, results = hybrid_retrieval(query_embedding, query)

# Output results with indices
for idx, res in zip(indices, results):
    print(f"Index: {idx} -> Result: {res}")

Index: 33 -> Result: {'query': 'machine learning', 'rank': 5.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'Supervised Machine Learning: Regression and Classification', 'combined_text': 'machine learning'}
Index: 37 -> Result: {'query': 'machine learning', 'rank': 9.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'What is Machine Learning? Types & Uses', 'combined_text': 'machine learning'}
Index: 38 -> Result: {'query': 'deep learning', 'rank': 1.0, 'type': 'article', 'keyword': 'deep learning', 'title': 'What is Deep Learning? - AWS', 'combined_text': 'deep learning'}
Index: 31 -> Result: {'query': 'machine learning', 'rank': 3.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'Machine Learning Tutorial', 'combined_text': 'machine learning'}
Index: 85 -> Result: {'query': 'Chapter Title', 'rank': nan, 'type': 'textbook', 'keyword': 'Chapter Keywords', 'title': 'Chapter Title', 'combined_text': 'Chapter Title'}


In [35]:
# Example usage
query = "Regression"
query_embedding = generate_swin_embedding(query)
# Call retrieval
indices, results = hybrid_retrieval(query_embedding, query)

# Output results with indices
for idx, res in zip(indices, results):
    print(f"Index: {idx} -> Result: {res}")

Index: 64 -> Result: {'query': 'data science', 'rank': 8.0, 'type': 'article', 'keyword': 'data science', 'title': 'What Is Data Science?', 'combined_text': 'data science'}
Index: 37 -> Result: {'query': 'machine learning', 'rank': 9.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'What is Machine Learning? Types & Uses', 'combined_text': 'machine learning'}
Index: 38 -> Result: {'query': 'deep learning', 'rank': 1.0, 'type': 'article', 'keyword': 'deep learning', 'title': 'What is Deep Learning? - AWS', 'combined_text': 'deep learning'}
Index: 31 -> Result: {'query': 'machine learning', 'rank': 3.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'Machine Learning Tutorial', 'combined_text': 'machine learning'}
Index: 85 -> Result: {'query': 'Chapter Title', 'rank': nan, 'type': 'textbook', 'keyword': 'Chapter Keywords', 'title': 'Chapter Title', 'combined_text': 'Chapter Title'}


In [36]:
# Example usage
query = "Classification"
query_embedding = generate_swin_embedding(query)
# Call retrieval
indices, results = hybrid_retrieval(query_embedding, query)

# Output results with indices
for idx, res in zip(indices, results):
    print(f"Index: {idx} -> Result: {res}")

Index: 33 -> Result: {'query': 'machine learning', 'rank': 5.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'Supervised Machine Learning: Regression and Classification', 'combined_text': 'machine learning'}
Index: 37 -> Result: {'query': 'machine learning', 'rank': 9.0, 'type': 'article', 'keyword': 'machine learning', 'title': 'What is Machine Learning? Types & Uses', 'combined_text': 'machine learning'}
Index: 38 -> Result: {'query': 'deep learning', 'rank': 1.0, 'type': 'article', 'keyword': 'deep learning', 'title': 'What is Deep Learning? - AWS', 'combined_text': 'deep learning'}
Index: 12 -> Result: {'query': 'Classification', 'rank': 3.0, 'type': 'article', 'keyword': 'Classification', 'title': 'CLASSIFICATION | English meaning - Cambridge Dictionary', 'combined_text': 'Classification'}
Index: 13 -> Result: {'query': 'Classification', 'rank': 4.0, 'type': 'article', 'keyword': 'Classification', 'title': 'Classification - Definition, Meaning & Synonyms', 'combined_